# Post-Hoc Explanations vs SENN — FashionMNIST (λ=1e-2, 5 concepts)

This notebook **visualises and compares** three explanation approaches on the same SENN model
trained on FashionMNIST with `robust_reg=1e-2` and `num_concepts=5`.

| Method | Type | Library |
|---|---|---|
| **SENN** (built-in) | Self-explaining (concepts × relevances) | custom |
| **Integrated Gradients** | Post-hoc gradient-based | `captum` |
| **LIME** | Post-hoc perturbation-based | `captum` |

**Pre-requisite:** Run the computation scripts before this notebook:
```bash
python run_integrated_gradients.py --config configs/fashion_mnist_lambda1e-2_c5_seed29.json
python run_lime.py --config configs/fashion_mnist_lambda1e-2_c5_seed29.json
```

**FashionMNIST class labels:**

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| T-shirt/top | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Ankle boot |

In [ ]:
# ─── Imports & constants ──────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from types import SimpleNamespace
from pathlib import Path

from senn.trainer import SENN_Trainer

plt.style.use('seaborn-v0_8-talk')
%matplotlib inline

FASHION_MNIST_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

CONFIG_PATH = 'configs/fashion_mnist_lambda1e-2_c5_seed29.json'
EXP_NAME    = 'fashion_mnist_lambda1e-2_c5_seed29'
POSTHOC_DIR = Path('results') / EXP_NAME / 'posthoc'
DEVICE      = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('Imports OK')

---
## 1. Load SENN model and pre-computed attributions

In [ ]:
# ─── Load SENN model ─────────────────────────────────────────────────────────
def load_senn(config_path, device='cpu'):
    with open(config_path, 'r') as f:
        config = json.load(f)
    config['device'] = device
    config['train'] = False
    config = SimpleNamespace(**config)
    checkpoint_path = Path('results') / config.exp_name / 'checkpoints' / 'best_model.pt'
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")
    trainer = SENN_Trainer(config)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    trainer.model.load_state_dict(checkpoint['model_state'])
    trainer.best_accuracy = checkpoint['best_accuracy']
    trainer.model.eval()
    print(f"SENN loaded — best valid acc: {checkpoint['best_accuracy']*100:.2f}%")
    return trainer

trainer     = load_senn(CONFIG_PATH, device=DEVICE)
model       = trainer.model
test_loader = trainer.test_loader
print(f'Device: {DEVICE}')

In [ ]:
# ─── Load pre-computed IG and LIME results ───────────────────────────────────
def load_posthoc(prefix, posthoc_dir):
    """Load attributions, predictions, labels, ablation drops, and meta for a method."""
    attrs  = torch.load(posthoc_dir / f'{prefix}_attributions.pt', map_location='cpu')
    preds  = torch.load(posthoc_dir / f'{prefix}_predictions.pt', map_location='cpu')
    labels = torch.load(posthoc_dir / f'{prefix}_labels.pt', map_location='cpu')
    drops  = np.load(posthoc_dir / f'{prefix}_ablation_drops.npy')
    with open(posthoc_dir / f'{prefix}_meta.json', 'r') as f:
        meta = json.load(f)
    return attrs, preds, labels, drops, meta

ig_attrs, ig_preds, ig_labels, ig_drops, ig_meta       = load_posthoc('ig', POSTHOC_DIR)
lime_attrs, lime_preds, lime_labels, lime_drops, lime_meta = load_posthoc('lime', POSTHOC_DIR)

print(f'IG:   {len(ig_labels)} samples, {ig_meta["total_time_s"]:.1f}s total')
print(f'LIME: {len(lime_labels)} samples, {lime_meta["total_time_s"]:.1f}s total')

---
## 2. Select representative samples

One sample per class for qualitative comparison.

In [ ]:
# ─── Pick one sample per class from the test set ─────────────────────────────
def get_one_per_class(loader, num_classes=10, device='cpu'):
    examples, labels = {}, {}
    for x, y in loader:
        for i in range(len(y)):
            lab = y[i].item()
            if lab not in examples:
                examples[lab] = x[i]
                labels[lab] = lab
            if len(examples) == num_classes:
                break
        if len(examples) == num_classes:
            break
    imgs = torch.stack([examples[i] for i in range(num_classes)]).to(device)
    labs = torch.tensor([labels[i] for i in range(num_classes)]).to(device)
    return imgs, labs

# We also need to find the indices of these samples in the saved attribution tensors
def find_per_class_indices(all_labels, num_classes=10):
    """Find the index of the first occurrence of each class in the label tensor."""
    indices = {}
    for i, lab in enumerate(all_labels.tolist()):
        if lab not in indices:
            indices[lab] = i
        if len(indices) == num_classes:
            break
    return [indices[c] for c in range(num_classes)]

per_class_imgs, per_class_labs = get_one_per_class(test_loader, device=DEVICE)
pc_indices = find_per_class_indices(ig_labels)

# Slice pre-computed attributions for per-class samples
ig_attr_pc   = ig_attrs[pc_indices]
lime_attr_pc = lime_attrs[pc_indices]
print(f'Per-class sample indices: {pc_indices}')

---
## 3. SENN built-in explanations

SENN provides concepts $h$ and relevances $\theta$. The prediction is:
$$\hat{y} = \text{softmax}(\theta^\top h)$$

In [ ]:
# ─── SENN explanations (live — single forward pass) ──────────────────────────
model.eval()
with torch.no_grad():
    y_pred_pc, (concepts_pc, relevances_pc), _ = model(per_class_imgs)
targets_pc = y_pred_pc.argmax(1)


def plot_senn_explanations(images, y_pred, concepts, relevances, class_names, ncols=5):
    n = len(images)
    pred_idx = y_pred.argmax(1)
    num_concepts = concepts.shape[1]
    concept_lim = max(abs(concepts.min().item()), abs(concepts.max().item())) + 0.1
    y_labels = [f'C{j+1}' for j in range(num_concepts - 1, -1, -1)]
    for g in range((n + ncols - 1) // ncols):
        start = g * ncols
        end = min(start + ncols, n)
        nc = end - start
        fig, axes = plt.subplots(3, nc, figsize=(3.2 * nc, 9))
        if nc == 1:
            axes = axes[:, np.newaxis]
        for col, idx in enumerate(range(start, end)):
            true_lab = class_names[idx] if idx < len(class_names) else '?'
            pred_lab = class_names[pred_idx[idx].item()]
            color = 'green' if true_lab == pred_lab else 'red'
            axes[0, col].imshow(images[idx].squeeze().cpu(), cmap='gray')
            axes[0, col].set_title(f'True: {true_lab}\nPred: {pred_lab}', fontsize=9, color=color)
            axes[0, col].axis('off')
            rs = relevances[idx, :, pred_idx[idx].item()].cpu()
            colors_r = ['b' if v > 0 else 'r' for v in rs.tolist()][::-1]
            axes[1, col].barh(np.arange(num_concepts), np.flip(rs.numpy()), color=colors_r)
            axes[1, col].set_yticks(np.arange(num_concepts))
            axes[1, col].set_yticklabels(y_labels, fontsize=8)
            axes[1, col].set_xlim(-1.1, 1.1)
            axes[1, col].axvline(0, color='k', lw=0.6, ls='--')
            if col == 0:
                axes[1, col].set_ylabel('Relevances \u03b8')
            cs = concepts[idx].flatten().cpu()
            colors_c = ['b' if v > 0 else 'r' for v in cs.tolist()][::-1]
            axes[2, col].barh(np.arange(num_concepts), np.flip(cs.numpy()), color=colors_c)
            axes[2, col].set_yticks(np.arange(num_concepts))
            axes[2, col].set_yticklabels(y_labels, fontsize=8)
            axes[2, col].set_xlim(-concept_lim, concept_lim)
            axes[2, col].axvline(0, color='k', lw=0.6, ls='--')
            if col == 0:
                axes[2, col].set_ylabel('Concepts h')
        plt.suptitle('SENN Built-in Explanations', fontweight='bold')
        plt.tight_layout()
        plt.show()

plot_senn_explanations(per_class_imgs, y_pred_pc, concepts_pc, relevances_pc, FASHION_MNIST_CLASSES)

---
## 4. Integrated Gradients — saliency maps

In [ ]:
# ─── Visualise IG saliency maps (from pre-computed attributions) ─────────────
def plot_saliency_maps(images, attributions, pred_labels, true_labels,
                       class_names, method_name, ncols=5):
    n = len(images)
    for g in range((n + ncols - 1) // ncols):
        start = g * ncols
        end = min(start + ncols, n)
        nc = end - start
        fig, axes = plt.subplots(2, nc, figsize=(3.2 * nc, 6))
        if nc == 1:
            axes = axes[:, np.newaxis]
        for col, idx in enumerate(range(start, end)):
            true_lab = class_names[true_labels[idx].item()]
            pred_lab = class_names[pred_labels[idx].item()]
            color = 'green' if true_lab == pred_lab else 'red'
            axes[0, col].imshow(images[idx].squeeze().cpu(), cmap='gray')
            axes[0, col].set_title(f'True: {true_lab}\nPred: {pred_lab}', fontsize=9, color=color)
            axes[0, col].axis('off')
            attr = attributions[idx].sum(dim=0).cpu().numpy()
            abs_max = max(abs(attr.min()), abs(attr.max())) + 1e-8
            axes[1, col].imshow(attr, cmap='seismic', vmin=-abs_max, vmax=abs_max)
            axes[1, col].set_title(method_name, fontsize=9)
            axes[1, col].axis('off')
        plt.suptitle(f'{method_name} — Saliency Maps', fontweight='bold')
        plt.tight_layout()
        plt.show()

plot_saliency_maps(per_class_imgs, ig_attr_pc, targets_pc, per_class_labs,
                   FASHION_MNIST_CLASSES, 'Integrated Gradients')

---
## 5. LIME — saliency maps

In [ ]:
# ─── Visualise LIME saliency maps (from pre-computed attributions) ────────────
plot_saliency_maps(per_class_imgs, lime_attr_pc, targets_pc, per_class_labs,
                   FASHION_MNIST_CLASSES, 'LIME')

---
## 6. Side-by-Side Comparison

For each class: original image | SENN relevance bars | IG heatmap | LIME heatmap.

In [ ]:
# ─── Three-way comparison ────────────────────────────────────────────────────
def plot_three_way_comparison(images, true_labels, y_pred, concepts, relevances,
                              ig_attrs, lime_attrs, class_names, ncols=5):
    n = len(images)
    pred_idx = y_pred.argmax(1)
    num_concepts = concepts.shape[1]
    y_labels = [f'C{j+1}' for j in range(num_concepts - 1, -1, -1)]
    for g in range((n + ncols - 1) // ncols):
        start = g * ncols
        end = min(start + ncols, n)
        nc = end - start
        fig, axes = plt.subplots(4, nc, figsize=(3.5 * nc, 13))
        if nc == 1:
            axes = axes[:, np.newaxis]
        for col, idx in enumerate(range(start, end)):
            true_lab = class_names[true_labels[idx].item()]
            pred_lab = class_names[pred_idx[idx].item()]
            color = 'green' if true_lab == pred_lab else 'red'
            # Row 0 — image
            axes[0, col].imshow(images[idx].squeeze().cpu(), cmap='gray')
            axes[0, col].set_title(f'True: {true_lab}\nPred: {pred_lab}', fontsize=9, color=color)
            axes[0, col].axis('off')
            # Row 1 — SENN relevances
            rs = relevances[idx, :, pred_idx[idx].item()].cpu()
            colors_r = ['b' if v > 0 else 'r' for v in rs.tolist()][::-1]
            axes[1, col].barh(np.arange(num_concepts), np.flip(rs.numpy()), color=colors_r)
            axes[1, col].set_yticks(np.arange(num_concepts))
            axes[1, col].set_yticklabels(y_labels, fontsize=8)
            axes[1, col].set_xlim(-1.1, 1.1)
            axes[1, col].axvline(0, color='k', lw=0.6, ls='--')
            if col == 0:
                axes[1, col].set_ylabel('SENN \u03b8', fontsize=10)
            # Row 2 — IG
            ig_map = ig_attrs[idx].sum(dim=0).cpu().numpy()
            abs_max_ig = max(abs(ig_map.min()), abs(ig_map.max())) + 1e-8
            axes[2, col].imshow(ig_map, cmap='seismic', vmin=-abs_max_ig, vmax=abs_max_ig)
            axes[2, col].axis('off')
            if col == 0:
                axes[2, col].set_ylabel('Int. Grad.', fontsize=10)
            # Row 3 — LIME
            lime_map = lime_attrs[idx].sum(dim=0).cpu().numpy()
            abs_max_lm = max(abs(lime_map.min()), abs(lime_map.max())) + 1e-8
            axes[3, col].imshow(lime_map, cmap='seismic', vmin=-abs_max_lm, vmax=abs_max_lm)
            axes[3, col].axis('off')
            if col == 0:
                axes[3, col].set_ylabel('LIME', fontsize=10)
        plt.suptitle('SENN vs Integrated Gradients vs LIME', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

plot_three_way_comparison(
    per_class_imgs, per_class_labs,
    y_pred_pc, concepts_pc, relevances_pc,
    ig_attr_pc, lime_attr_pc,
    FASHION_MNIST_CLASSES
)

---
## 7. Computational Cost Comparison

Timing from the pre-computed runs (wall-clock, full test set).

In [ ]:
# ─── Computational cost (from saved meta) ────────────────────────────────────
# SENN cost: single forward pass on the test set for reference
import time
model.eval()
n_senn = 0
t0 = time.perf_counter()
with torch.no_grad():
    for x, _ in test_loader:
        _ = model(x.float().to(DEVICE))
        n_senn += len(x)
senn_time_per_sample = (time.perf_counter() - t0) / n_senn

ig_time_per_sample   = ig_meta['time_per_sample_s']
lime_time_per_sample = lime_meta['time_per_sample_s']

print(f'{"Method":<25} {"s / sample":>12} {"Fwd passes":>14}')
print('\u2500' * 55)
print(f'{"SENN (built-in)":<25} {senn_time_per_sample:>12.5f} {"1":>14}')
print(f'{"Integrated Gradients":<25} {ig_time_per_sample:>12.5f} {"~" + str(ig_meta.get("n_steps", 50)):>14}')
print(f'{"LIME":<25} {lime_time_per_sample:>12.5f} {"~" + str(lime_meta.get("n_lime_samples", 500)):>14}')

In [ ]:
# ─── Cost bar chart ──────────────────────────────────────────────────────────
methods = ['SENN\n(built-in)', 'Integrated\nGradients', 'LIME']
times = [senn_time_per_sample, ig_time_per_sample, lime_time_per_sample]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, times, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Seconds per sample')
ax.set_title('Computational Cost Comparison', fontweight='bold')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15,
            f'{t:.4f}s', ha='center', va='bottom', fontsize=10)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Faithfulness — Feature Ablation (top-20% masking)

**IG / LIME:** top-20% most important pixels (by |attribution|) replaced with dataset mean.  
**SENN:** concept with highest |θ| zeroed out.  
Metric: drop in softmax confidence for the predicted class.

IG and LIME ablation drops were computed in the scripts.  
SENN ablation is computed below (fast — no attributions needed, just a forward pass).

In [ ]:
# ─── SENN concept ablation (computed live) ───────────────────────────────────
all_senn_drops = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.float().to(DEVICE)
        y = y.long().to(DEVICE)

        y_pred, (concepts, relevances), _ = model(x)
        preds = y_pred.argmax(1)
        probs_orig = torch.softmax(y_pred, dim=1)
        conf_orig = probs_orig[torch.arange(len(preds)), preds]

        # Zero-out the concept with highest |θ| for the predicted class
        concepts_abl = concepts.clone()
        for i in range(len(x)):
            rel_for_pred = relevances[i, :, preds[i].item()].abs()
            top_concept = rel_for_pred.argmax().item()
            concepts_abl[i, top_concept, :] = 0.0

        y_pred_abl = model.aggregator(concepts_abl, relevances)
        probs_abl = torch.softmax(y_pred_abl, dim=1)
        conf_abl = probs_abl[torch.arange(len(preds)), preds]
        all_senn_drops.append((conf_orig - conf_abl).cpu().numpy())

all_senn_drops = np.concatenate(all_senn_drops)
print(f'SENN concept ablation: {len(all_senn_drops)} samples, '
      f'mean drop = {all_senn_drops.mean():.4f}')

In [ ]:
# ─── Ablation results ────────────────────────────────────────────────────────
print(f'{"Method":<30} {"Mean drop":>12} {"Std":>10} {"N samples":>10}')
print('\u2500' * 65)
print(f'{"Integrated Gradients (20% px)":<30} {ig_drops.mean():>12.4f} {ig_drops.std():>10.4f} {len(ig_drops):>10}')
print(f'{"LIME (20% px)":<30} {lime_drops.mean():>12.4f} {lime_drops.std():>10.4f} {len(lime_drops):>10}')
print(f'{"SENN (top-1 concept)":<30} {all_senn_drops.mean():>12.4f} {all_senn_drops.std():>10.4f} {len(all_senn_drops):>10}')

In [ ]:
# ─── Ablation bar chart ──────────────────────────────────────────────────────
methods_abl = ['Integrated\nGradients\n(top-20% px)', 'LIME\n(top-20% px)',
               'SENN\n(top-1 concept)']
mean_drops = [ig_drops.mean(), lime_drops.mean(), all_senn_drops.mean()]
std_drops  = [ig_drops.std(),  lime_drops.std(),  all_senn_drops.std()]
colors_abl = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(methods_abl, mean_drops, yerr=std_drops, color=colors_abl,
              edgecolor='black', linewidth=0.5, capsize=5)
ax.set_ylabel('Mean Confidence Drop')
ax.set_title('Faithfulness: Ablation Test', fontweight='bold')
for bar, m in zip(bars, mean_drops):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Summary Table

In [ ]:
# ─── Summary ─────────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Method': ['SENN (built-in)', 'Integrated Gradients', 'LIME'],
    'Type': ['Self-explaining', 'Post-hoc (gradient)', 'Post-hoc (perturbation)'],
    'Time/sample (s)': [senn_time_per_sample, ig_time_per_sample, lime_time_per_sample],
    'Fwd passes': [1, ig_meta.get('n_steps', 50), lime_meta.get('n_lime_samples', 500)],
    'Mean conf. drop': [all_senn_drops.mean(), ig_drops.mean(), lime_drops.mean()],
})
summary = summary.round(4)
print(summary.to_string(index=False))
summary